# Speech Emotion Fine-Tuning — Colab / Kaggle

Run speech emotion recognition experiments on RESD dataset.  
Tested on T4 GPU (Colab free tier / Kaggle).

**Steps:** install deps → clone repo → pick config → train → visualise

## 1. GPU check

In [ ]:
import subprocess, sys, os
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No GPU found — switch runtime to GPU.')

## 2. Install dependencies

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchaudio', 'torchcodec',
    'transformers>=4.40', 'datasets>=2.18', 'peft>=0.10',
    'scikit-learn', 'matplotlib', 'seaborn',
    'soundfile', 'pyyaml', 'tqdm',
], check=True)
print('Done.')

## 3. Get the code

**Option A — clone from GitHub:**

In [ ]:
REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print('Working directory:', os.getcwd())

**Option B — Google Drive mount** (Colab only):

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# os.chdir('/content/drive/MyDrive/speech_emo_finetune')
# if os.getcwd() not in sys.path:
#     sys.path.insert(0, os.getcwd())

## 4. Choose experiment config

In [ ]:
# Available configs — pick one:
#   wavlm_base_head    (fastest, ~8 min on T4)
#   wavlm_base_lora    (~12 min)
#   wavlm_base_full    (~20 min)
#   hubert_large_head  (~15 min)
#   hubert_large_lora  (~25 min)
#   hubert_large_topn  (~25 min)
#   hubert_large_lstm  (~10 min, frozen backbone)
#   wav2vec2_xlsr_head (~15 min)
#   wav2vec2_xlsr_lora (~25 min)

CONFIG = 'configs/wavlm_base_head.yaml'
print('Selected config:', CONFIG)

## 5. Train

Runs directly in the notebook — output (tqdm bars, epoch table) appears in real time.

In [ ]:
import random
import torch
import numpy as np
from transformers import AutoFeatureExtractor

from src.config import load_config
from src.dataset import get_dataloaders
from src.models import build_model
from src.trainer import Trainer

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

config = load_config(CONFIG)
set_seed(config.seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}', flush=True)

processor = AutoFeatureExtractor.from_pretrained(config.model_name)
train_loader, dev_loader, test_loader = get_dataloaders(config, processor)
print(f'Train: {len(train_loader.dataset)} | Dev: {len(dev_loader.dataset)} | Test: {len(test_loader.dataset)}', flush=True)

model = build_model(config)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Params: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)', flush=True)

trainer = Trainer(model, config, train_loader, dev_loader, test_loader, device)
test_metrics = trainer.fit()

## 6. Training curves

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt
import yaml

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
metrics_path = pathlib.Path(cfg.get('output_dir', 'outputs/experiment')) / 'metrics.jsonl'

records = [json.loads(l) for l in open(metrics_path) if 'epoch' in json.loads(l)]
epochs     = [r['epoch'] for r in records]
train_loss = [r['train_loss'] for r in records]
dev_f1     = [r['dev_f1_weighted'] for r in records]
dev_acc    = [r['dev_accuracy'] for r in records]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, train_loss, marker='o')
axes[0].set_title('Train Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(epochs, dev_f1, marker='o', label='F1 weighted')
axes[1].plot(epochs, dev_acc, marker='s', linestyle='--', label='Accuracy')
axes[1].set_title('Dev Metrics')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.suptitle(cfg.get('run_name', CONFIG))
plt.tight_layout()
plt.show()

## 7. Test results + confusion matrix

In [ ]:
import numpy as np
import seaborn as sns

all_records = [json.loads(l) for l in open(metrics_path)]
test_rec = [r for r in all_records if r.get('split') == 'test'][-1]

print(f"accuracy          {test_rec['accuracy']:.4f}")
print(f"weighted_accuracy {test_rec['weighted_accuracy']:.4f}")
print(f"f1_macro          {test_rec['f1_macro']:.4f}")
print(f"f1_weighted       {test_rec['f1_weighted']:.4f}")
print()
print(f"{'Class':12s} {'P':>6} {'R':>6} {'F1':>6} {'n':>5}")
for cls, vals in test_rec['per_class'].items():
    print(f"{cls:12s} {vals['precision']:6.3f} {vals['recall']:6.3f} {vals['f1']:6.3f} {int(vals['support']):5d}")

cm = np.array(test_rec['confusion_matrix'])
label_names = list(test_rec['per_class'].keys())

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_names, yticklabels=label_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix — {cfg.get("run_name", "")}')
plt.tight_layout()
plt.show()

## 8. Compare all experiments

In [ ]:
import pandas as pd

results = []
for mf in sorted(pathlib.Path('outputs').rglob('metrics.jsonl')):
    recs = [json.loads(l) for l in open(mf)]
    test_recs = [r for r in recs if r.get('split') == 'test']
    if not test_recs:
        continue
    tr = test_recs[-1]
    results.append({'run': mf.parent.name, **{k: tr[k] for k in ('accuracy', 'weighted_accuracy', 'f1_macro', 'f1_weighted')}})

if results:
    df = pd.DataFrame(results).set_index('run').sort_values('f1_weighted', ascending=False)
    display(df.style.format('{:.4f}').background_gradient(cmap='YlGn'))

    df[['accuracy', 'f1_macro', 'f1_weighted']].plot(kind='bar', figsize=(10, 5))
    plt.title('Experiment comparison')
    plt.ylabel('Score')
    plt.xticks(rotation=30, ha='right')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    print('No completed experiments found in outputs/')